In [1]:
import boto3

# How Streaming Works
Instead of waiting for the entire response to be generated, streaming sends back pieces of text as soon as they're available. 
Here's how the flow changes:

When you call `converse_stream`, you immediately get back an initial response that contains a `stream object`. This stream is a generator that yields events as the model generates text. Each event contains a small chunk of the overall response.

# Basic Implementation
Here's how to use converse_stream in your code:

` messages = []`

`add_user_message(messages, "Write a 1 sentence description of a fake database")`

`response = client.converse_stream(messages=messages, modelId=model_id)`


`for event in response["stream"]:`

   ` print(event)`

This will print out all the different events as they arrive. You'll see the response come in chunks rather than all at once.

# Understanding Stream Events
The stream yields several types of events, each serving a different purpose:
For basic text generation, you only need to care about `contentBlockDelta` events. These contain the actual generated text chunks that you want to display to users.

The events always arrive in a predictable order: `messageStart`, multiple `contentBlockDelta` events containing your text, then `contentBlockStop`, `messageStop`, and finally `metadata`.

In [4]:
client = boto3.client("bedrock-runtime", region_name="us-west-2")
model_id = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"


def add_user_message(messages, text):
    user_message = {"role": "user", "content": [{"text": text}]}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": [{"text": text}]}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {"temperature": temperature},
    }

    if system:
        params["system"] = [{"text": system}]

    response = client.converse(**params)

    return response["output"]["message"]["content"][0]["text"]

In [14]:
messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database")
response = client.converse_stream(messages=messages, modelId=model_id)

text = ""
for event in response["stream"]:
    if "contentBlockDelta" in event:
        chunk = event["contentBlockDelta"]["delta"]["text"]
        print(chunk, end="")
        text += chunk

print("\n\nTotal Message:\n" + text)

"SecureVault Pro is a fictional database management system that promises to organize your data with unparalleled efficiency while using proprietary quantum encryption algorithms that don't actually exist."

Total Message:
"SecureVault Pro is a fictional database management system that promises to organize your data with unparalleled efficiency while using proprietary quantum encryption algorithms that don't actually exist."


# Extracting the Text
To get just the generated text from each chunk, filter for contentBlockDelta events and extract the text:

The `end="" `parameter removes the automatic newline that Python's print function adds, making the streaming text appear more naturally.